::: {.claves}
[En breve · {{< meta tipo >}}]{.etiqueta}

{{< meta resumen >}}
:::

In [1]:
#| label: setup
#| include: false
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "_quarto.yml").exists())
sys.path.insert(0, str(ROOT))

import polars as pl
from IPython.display import Markdown, display
from plotly.subplots import make_subplots
from common.files import read_processed
from common.paths import topic_dir
from common.plotting import ACCENT, GREY, PALETTE, add_source, init_plotly, new_figure
init_plotly()

TOPIC = topic_dir()
stocks = read_processed(TOPIC, "p01_stocks")
comp = read_processed(TOPIC, "p01_components")
births = read_processed(TOPIC, "p01_births_mother")
deaths = read_processed(TOPIC, "p01_deaths_group")
checks = read_processed(TOPIC, "p01_checks")

def s(nac: str, nacim: str) -> pl.DataFrame:
    """Serie a 1 de enero de un cruce nacionalidad × país de nacimiento."""
    return stocks.filter(pl.col("nacionalidad") == nac, pl.col("nacimiento") == nacim).select("anio", "poblacion")

def miles(x: float) -> str:
    return f"{x:,.0f}".replace(",", ".")

def show(df: pl.DataFrame, decimals: int = 0) -> Markdown:
    """Tabla en Markdown: miles con punto, decimales con coma, «—» si no hay dato."""
    def fmt(v):
        if v is None:
            return "—"
        if isinstance(v, (int, float)):
            t = f"{v:,.{decimals if isinstance(v, float) else 0}f}"
            return t.replace(",", "X").replace(".", ",").replace("X", ".").replace("-", "−")
        return str(v)
    num = [c for c, t in df.schema.items() if t.is_numeric()]
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---:" if c in num else ":---" for c in df.columns) + "|"
    body = ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.iter_rows()]
    return Markdown("\n".join([head, sep, *body]))

def place_source(fig, y: float = -0.18, bottom: int = 80) -> None:
    """Coloca la nota de fuente más abajo en figuras con subgráficos."""
    fig.layout.annotations[-1].y = y
    fig.layout.annotations[-1].yanchor = "top"
    fig.update_layout(margin=dict(b=bottom))

SOURCE = "Fuente: INE (ECP, MNP, Estadística de Migraciones). Elaboración propia."
# Gobiernos: cada año se asigna al gobierno que lo ocupó en su mayor parte
GOV = [("Aznar", 2002, 2003), ("Zapatero", 2004, 2011), ("Rajoy", 2012, 2017), ("Sánchez", 2018, 2025)]

def shade_governments(fig, y_top_note: bool = True) -> None:
    for k, (name, a, b) in enumerate(GOV):
        fig.add_vrect(x0=a - 0.5, x1=b + 0.5, fillcolor="rgba(127,127,127,0.08)" if k % 2 else "rgba(0,0,0,0)",
                      line_width=0, layer="below")
        if y_top_note:
            fig.add_annotation(x=(a + b) / 2, y=0.99, yref="paper", text=name, showarrow=False,
                               font=dict(size=11, color=GREY), yanchor="top")

## La pregunta

España tenía 49,6 millones de habitantes el 1 de enero de 2026, el máximo de su historia, y la
inmigración es uno de los temas centrales de la campaña de las elecciones generales del 29 de
noviembre. Esta entrada responde a una pregunta previa a cualquier valoración: **¿de dónde sale
el crecimiento de la población?** ¿De los nacimientos, de la llegada de personas del extranjero,
o de ambos?

## Lo que se debate

Estas declaraciones recientes muestran por qué la pregunta importa. **Motivan** el análisis,
pero ninguna de ellas formula exactamente las afirmaciones que se contrastan más abajo, que son
del blog.

- **Pedro Sánchez** (PSOE, presidente del Gobierno), 15-I-2026: «En España no sobra nadie, no
  sobra nadie. Al contrario, nos falta gente»
  ([transcripción, La Moncloa](https://www.lamoncloa.gob.es/presidente/intervenciones/Documents/2026/260115TranscripcionPGSpainInvestorsDay.pdf)).
- **Santiago Abascal** (Vox), mitin en Guadalajara, 7-XI-2026: «Ahora mismo no queremos que venga
  nadie», «Ha venido demasiada gente»
  ([La Bandera](https://labandera.es/abascal-endurece-su-discurso-contra-la-inmigracion-en-guadalajara-ahora-mismo-no-queremos-que-venga-nadie/)).
- **Alberto Núñez Feijóo** (PP), 10-XI-2026: «el caos y el efecto llamada están provocando una
  invasión de inmigración irregular en nuestro país»
  ([The Objective](https://theobjective.com/espana/politica/2026-11-10/nunez-feijoo-derogaremos-ley-memoria-democratica-ley-de-bildu-y-sanchez/)).
- **INE**, nota de prensa de la Estadística Continua de Población (4.º trimestre de 2025): «El
  crecimiento poblacional de España se debió al incremento de personas nacidas en el extranjero»
  ([INE](https://ine.es/dyngs/Prensa/ECP4T25.htm)).

::: {.callout-note collapse="true" title="Afirmaciones del debate que quedan fuera de esta entrada, y por qué"}
- **«Sustitución» o «reemplazo» de la población como plan deliberado.** Es una afirmación sobre
  intenciones, no sobre datos, y no se puede contrastar con estadísticas demográficas. La
  literatura académica la clasifica como teoría conspirativa (la del «Gran Reemplazo»).
- **«Nos falta gente», «sobra gente», «no cabe nadie más».** Son juicios normativos o sobre la
  capacidad del país: para contrastarlos haría falta un criterio medible (vacantes de empleo,
  vivienda, servicios), que es objeto de otras entradas.
- **«Cinco millones de personas en cinco años»** y el **«efecto llamada»**. La primera es
  falsable, pero trata de **flujos de llegada**, no del crecimiento; la segunda es una afirmación
  causal. Quedan para la entrada sobre migraciones.
:::

## Conceptos

Cuatro ideas de la estadística demográfica bastan para seguir el análisis.

**Población residente.** Las personas que viven habitualmente en España, sea cual sea su
nacionalidad. El INE la estima cada trimestre en la *Estadística Continua de Población* (ECP), a
partir del padrón y de otros registros administrativos.

**Stocks y flujos.** Un *stock* es cuántas personas hay en una fecha (la población a 1 de enero).
Un *flujo* es cuántas entran o salen en un periodo (nacimientos, defunciones, llegadas y salidas).
Muchas confusiones del debate público nacen de comparar uno con otro.

**La ecuación compensadora.** Toda variación de la población se debe a cuatro flujos:
$$
\underbrace{P_{t+1} - P_t}_{\text{crecimiento}} =
\underbrace{(N_t - D_t)}_{\text{saldo natural}} + \underbrace{(I_t - E_t)}_{\text{saldo migratorio}},
$$
donde $N$ son los nacimientos, $D$ las defunciones, $I$ las inmigraciones desde el extranjero y
$E$ las emigraciones hacia el extranjero en el año $t$. El **saldo natural** (o vegetativo) es
negativo cuando mueren más personas de las que nacen.

**Nacionalidad y país de nacimiento.** Son dos formas distintas de medir el «origen»:

- la **nacionalidad** es una condición jurídica que cambia: un extranjero que obtiene la
  nacionalidad española pasa a contar como español;
- el **país de nacimiento** es un hecho fijo: quien nació en el extranjero sigue contando como
  nacido en el extranjero toda su vida, y el hijo de padres extranjeros nacido en España cuenta
  como nacido en España.

::: {.callout-note collapse="true" title="Más detalle: medición de los flujos y adquisición de la nacionalidad"}
- En la práctica la ecuación compensadora no cuadra exactamente: la población, los nacimientos y
  las migraciones proceden de estadísticas distintas, con fuentes y revisiones distintas. La
  diferencia es un **residuo** que hay que mostrar, no esconder (ver *Cómo lo comprobamos*).
- Los nacimientos y las defunciones proceden del *Movimiento Natural de la Población* (MNP), que
  explota los registros civiles; las migraciones, de la *Estadística de Migraciones*, basada en las
  altas y bajas del padrón por cambio de residencia con el extranjero.
- **Adquisición de la nacionalidad por residencia** (artículo 22 del Código Civil): exige, con
  carácter general, diez años de residencia legal y continuada; cinco para refugiados; dos para
  nacionales de países iberoamericanos, Andorra, Filipinas, Guinea Ecuatorial y Portugal, y para
  sefardíes; y uno, entre otros, para quienes nacieron en España. Los dos años se aplican a los
  nacionales *de origen* de esos países, lo que contribuye a que buena parte de las
  nacionalizaciones sean de personas de origen americano, aunque el primer país en 2025 fue
  Marruecos.
:::

## Las afirmaciones que contrastamos

Formuladas por el blog para que sean falsables, a partir de lo que se repite en el debate.

**A1. La población de España crece solo por la llegada de población del extranjero:** el saldo
natural es negativo, y todo el crecimiento procede del saldo migratorio.

- *A1a*, con los flujos de la ecuación compensadora: saldo natural y saldo migratorio, año a año.
- *A1b*, con los stocks: población nacida en España y nacida en el extranjero.

**A2. La población de origen español disminuye mientras aumenta la de origen extranjero.**

- *A2a*: «origen» = país de nacimiento.
- *A2b*: «origen» = nacionalidad.

**Veredicto.** Las afirmaciones están en presente, así que se juzgan en la **última década**
(2016–2025); el resto de la serie muestra si el veredicto depende de la etapa. Los criterios se
fijaron antes de procesar los datos, aunque ya se conocían titulares como la caída de la población
nacida en España que señala el INE:

::: {.callout-note collapse="true" title="Criterios del veredicto"}
- **A1** *se sostiene* si el saldo natural es ≤ 0 en todos los años de la década y el crecimiento
  no supera al saldo migratorio en ninguno; *con matices* si se cumple en al menos siete de diez
  años o solo con una de las dos lecturas (A1a, A1b); *no se sostiene* si el saldo natural es
  positivo en la mayoría de los años.
- **A2** *se sostiene* si, entre el 1-I-2016 y el 1-I-2026, el grupo «español» disminuye y el
  «extranjero» aumenta con las dos definiciones; *con matices* si se cumple solo con una; *no se
  sostiene* si no se cumple con ninguna.
:::

## Cómo lo comprobamos

| Dato | Estadística del INE | Periodo |
|---|---|---|
| Población a 1 de enero, por nacionalidad y país de nacimiento | ECP, tabla 79273 | 2002–2026 |
| Nacimientos y defunciones | MNP, tablas 6525 y 6562 | 1975–2024 |
| Nacimientos y defunciones de 2025 | Estimaciones experimentales EMN (46682) y EDeS (62277) | 2025 |
| Saldo migratorio con el extranjero | Estadística de Migraciones, tablas 24310 y 69760 | 2008–2024 |
| Nacimientos según la madre; defunciones según el fallecido | MNP, tablas 58929, 58930, 58701 y 70648; IDB 2776 | 2002–2024 |

**Decisiones que condicionan el resultado**

1. **Fecha de referencia.** La población se toma a 1 de enero; el crecimiento del año $t$ es la
   diferencia entre el 1 de enero de $t+1$ y el de $t$.
2. **Cambio de estadística de migraciones en 2021.** El INE sustituyó la *Estadística de
   Migraciones* por la *Estadística de Migraciones y Cambios de Residencia*. Para 2021, la antigua
   da un saldo de 148.070 personas y la nueva de 191.094: se usa la antigua hasta 2020 y la nueva
   desde 2021.
3. **Antes de 2008** no hay saldo migratorio publicado en esta serie: se calcula como
   *crecimiento − saldo natural* (saldo **implícito**), que incluye el residuo.
4. **2025.** Aún no hay datos definitivos del MNP: los nacimientos y defunciones de 2025 son
   **estimaciones experimentales** del INE.
5. **Revisiones.** La ECP se revisa: la población a 1-I-2026 es hoy de 49.596.499 personas, frente
   a 49.570.725 en la primera publicación. Se usa siempre el último dato.

::: {.callout-note collapse="true" title="Comprobaciones de los datos"}
- Nuestro saldo natural coincide exactamente con el *crecimiento vegetativo* publicado por el
  INE (tabla 6566) en todos los años disponibles.
- Por nacionalidad y por país de nacimiento, los grupos suman la población total en todas las
  fechas (diferencias de redondeo del INE inferiores a 10 personas).
- El **residuo** de la ecuación compensadora (crecimiento − saldo natural − saldo migratorio) es
  de unas 105.000–126.000 personas al año entre 2008 y 2011, y inferior a 16.000 desde 2012. La
  tabla lo muestra año a año. No es una validación independiente: las dos estadísticas parten
  del padrón.
:::

In [2]:
#| label: tbl-residuo
#| tbl-cap: "Ecuación compensadora año a año (personas). Residuo = crecimiento − saldo natural − saldo migratorio."
#| code-fold: true
show(comp.filter(pl.col("anio").is_between(2008, 2024))
     .select(Año=pl.col("anio").cast(pl.String), Crecimiento="crecimiento", **{"Saldo natural": "saldo_natural",
             "Saldo migratorio": "saldo_migratorio", "Residuo": "residuo"}))

| Año | Crecimiento | Saldo natural | Saldo migratorio | Residuo |
|:---|---:|---:|---:|---:|
| 2008 | 570.333 | 133.455 | 310.641 | 126.237 |
| 2009 | 247.350 | 110.064 | 12.845 | 124.441 |
| 2010 | 180.554 | 104.528 | −42.675 | 118.701 |
| 2011 | 151.041 | 84.088 | −37.698 | 104.651 |
| 2012 | −105.566 | 51.698 | −142.552 | −14.712 |
| 2013 | −216.906 | 35.296 | −251.531 | −671 |
| 2014 | −70.022 | 31.765 | −94.976 | −6.811 |
| 2015 | −6.838 | −2.278 | −1.761 | −2.799 |
| 2016 | 78.509 | −28 | 87.422 | −8.885 |
| 2017 | 147.677 | −31.342 | 163.272 | 15.747 |
| 2018 | 273.881 | −54.944 | 334.158 | −5.333 |
| 2019 | 399.099 | −58.086 | 454.232 | 2.953 |
| 2020 | 82.748 | −152.461 | 219.357 | 15.852 |
| 2021 | 85.929 | −113.364 | 191.094 | 8.199 |
| 2022 | 598.634 | −135.166 | 727.005 | 6.795 |
| 2023 | 534.334 | −115.468 | 642.296 | 7.506 |
| 2024 | 508.602 | −118.113 | 626.268 | 447 |

## Lo que dicen los datos

### A1a · Desde 2015 mueren más personas de las que nacen

En España nacen menos niños que hace dos décadas y, con el envejecimiento, mueren más personas. Las dos curvas
se cruzaron en **2015**: por primera vez en la serie que comienza en 1975, el saldo natural fue
negativo, y lo ha sido todos los años desde entonces.

In [3]:
#| label: fig-nacimientos-defunciones
#| fig-cap: "Nacimientos y defunciones en España, 1975–2025 (2025: estimación experimental)."
long = comp.filter(pl.col("anio") <= 2025)
fig = new_figure(y_title="Personas", height=380)
fig.add_scatter(x=long["anio"], y=long["nacimientos"], name="Nacimientos", mode="lines", line=dict(color=PALETTE[1], width=2.5))
fig.add_scatter(x=long["anio"], y=long["defunciones"], name="Defunciones", mode="lines", line=dict(color=ACCENT, width=2.5))
fig.add_vline(x=2015, line=dict(color=GREY, dash="dot"))
fig.add_annotation(x=2014.5, y=640_000, text="2015: el saldo natural<br>pasa a ser negativo", showarrow=False,
                   xanchor="right", font=dict(size=11, color=GREY))
add_source(fig, SOURCE)
fig.show()

Con el saldo natural negativo, todo el crecimiento procede del saldo migratorio. La figura descompone
el crecimiento de cada año en sus dos componentes.

In [4]:
#| label: fig-componentes
#| fig-cap: "Componentes del crecimiento de la población, 2002–2025. Barras: saldo natural y saldo migratorio (antes de 2008 y en 2025, implícito: crecimiento − saldo natural). Línea: crecimiento observado."
c = comp.filter(pl.col("anio").is_between(2002, 2025)).with_columns(
    mig_oficial=pl.col("saldo_migratorio"),
    mig_implicito=pl.when(pl.col("saldo_migratorio").is_null()).then(pl.col("saldo_migratorio_implicito")),
)
fig = new_figure(y_title="Personas al año", height=430, barmode="relative")
fig.add_bar(x=c["anio"], y=c["saldo_natural"], name="Saldo natural", marker_color=PALETTE[1])
fig.add_bar(x=c["anio"], y=c["mig_oficial"], name="Saldo migratorio", marker_color=ACCENT)
# base explícita: el implícito se apila sobre el saldo natural positivo (o sobre 0 si es negativo)
fig.add_bar(x=c["anio"], y=c["mig_implicito"], base=c["saldo_natural"].clip(lower_bound=0),
            name="Saldo migratorio (implícito)", marker_color=ACCENT, opacity=0.4)
fig.add_scatter(x=c["anio"], y=c["crecimiento"], name="Crecimiento", mode="lines+markers", line=dict(color="#555", width=2))
shade_governments(fig)
add_source(fig, SOURCE)
fig.show()

In [5]:
#| label: tbl-a1a
#| tbl-cap: "Última década: saldo natural, saldo migratorio y crecimiento (personas)."
#| code-fold: true
dec = comp.filter(pl.col("anio").is_between(2016, 2025))
show(dec.select(Año=pl.col("anio").cast(pl.String), **{"Saldo natural": "saldo_natural",
                "Saldo migratorio": "saldo_migratorio", "Crecimiento": "crecimiento"}))

| Año | Saldo natural | Saldo migratorio | Crecimiento |
|:---|---:|---:|---:|
| 2016 | −28 | 87.422 | 78.509 |
| 2017 | −31.342 | 163.272 | 147.677 |
| 2018 | −54.944 | 334.158 | 273.881 |
| 2019 | −58.086 | 454.232 | 399.099 |
| 2020 | −152.461 | 219.357 | 82.748 |
| 2021 | −113.364 | 191.094 | 85.929 |
| 2022 | −135.166 | 727.005 | 598.634 |
| 2023 | −115.468 | 642.296 | 534.334 |
| 2024 | −118.113 | 626.268 | 508.602 |
| 2025 | −123.455 | — | 468.202 |

In [6]:
#| include: false
dec_ok_nat = dec.filter(pl.col("saldo_natural") <= 0).height
dec_ok_mig = dec.drop_nulls("saldo_migratorio").filter(pl.col("crecimiento") <= pl.col("saldo_migratorio")).height
dec_n_mig = dec.drop_nulls("saldo_migratorio").height
n2016 = dec.filter(pl.col("anio") == 2016)["saldo_natural"].item()

::: {.resultado}
**A1a se sostiene en la última década.** El saldo natural fue negativo los diez años de 2016 a 2025
(en 2016, por solo 28 personas; ya lo había sido en 2015), y en los nueve años con saldo migratorio
publicado el crecimiento fue siempre menor que ese saldo. **Antes no era así:** entre 2002 y 2014 el
saldo natural fue positivo todos los años. La inmigración aportaba la mayor parte del crecimiento hasta
2008, y entre 2010 y 2015 el saldo migratorio fue negativo.
:::

### A1b · La población nacida en España disminuye cada año desde 2016

Los stocks cuentan la misma historia desde otro ángulo. La población nacida en España alcanzó su
máximo el 1-I-2013 (40,55 millones) y desde entonces baja cada año, con más fuerza desde 2016; la nacida en el extranjero
ha pasado de 5,9 a 10 millones en la misma década.

In [7]:
#| label: fig-stocks-nacimiento
#| fig-cap: "Población a 1 de enero según país de nacimiento, 2002–2026 (millones)."
esp, ext = s("Total", "España"), s("Total", "Extranjero")
fig = make_subplots(rows=1, cols=2, subplot_titles=("Nacida en España", "Nacida en el extranjero"))
fig.add_scatter(x=esp["anio"], y=esp["poblacion"] / 1e6, mode="lines+markers", line=dict(color=PALETTE[1]), showlegend=False, row=1, col=1)
fig.add_scatter(x=ext["anio"], y=ext["poblacion"] / 1e6, mode="lines+markers", line=dict(color=ACCENT), showlegend=False, row=1, col=2)
fig.update_yaxes(tickformat=",.1f")
fig.update_layout(height=360, margin=dict(t=50))
add_source(fig, SOURCE)
place_source(fig)
fig.show()

In [8]:
#| include: false
e16, e26 = (esp.filter(pl.col("anio") == y)["poblacion"].item() for y in (2016, 2026))
x16, x26 = (ext.filter(pl.col("anio") == y)["poblacion"].item() for y in (2016, 2026))
esp_down_every_year = (esp.filter(pl.col("anio") >= 2016)["poblacion"].diff().drop_nulls() < 0).all()

::: {.resultado}
**A1b se sostiene.** Entre el 1-I-2016 y el 1-I-2026 la población nacida en España bajó en unas
950.000 personas (de 40,51 a 39,56 millones), en todos los años del periodo, y la nacida en el
extranjero subió en 4,1 millones (de 5,91 a 10,04). Todo el crecimiento de la década corresponde
a personas nacidas fuera.
:::

::: {.aviso}
«Nacido en el extranjero» no es lo mismo que «extranjero» ni que «llegado recientemente»: incluye a
españoles nacidos fuera y a personas que llevan décadas en España. Y los hijos de inmigrantes
nacidos en España cuentan como nacidos en España.
:::

### A2 · ¿Disminuye la población de origen español? Depende de cómo se mida

Con el país de nacimiento, sí (A2a). Con la nacionalidad, no: la población con nacionalidad
española **aumenta** entre 2016 y 2026 (no todos los años: bajó en 2017, 2020 y 2021, y casi todo
el aumento es de 2023–2025). Las nacionalizaciones trasladan al grupo «español» a personas nacidas
fuera: entre unas 66.000 (2017) y 300.000 (2025) al año según la
[estadística de adquisiciones de nacionalidad del INE](https://ine.es/dyngs/Prensa/es/EANER2025.htm).

In [9]:
#| label: fig-definiciones
#| fig-cap: "Variación de la población entre el 1-I-2016 y el 1-I-2026, según cómo se defina el origen (millones de personas)."
def change(nac, nacim):
    d = s(nac, nacim)
    return (d.filter(pl.col("anio") == 2026)["poblacion"].item() - d.filter(pl.col("anio") == 2016)["poblacion"].item()) / 1e6
rows = [
    ("Por país de nacimiento", "Nacidos en España", change("Total", "España")),
    ("Por país de nacimiento", "Nacidos en el extranjero", change("Total", "Extranjero")),
    ("Por nacionalidad", "Nacionalidad española", change("Española", "Total")),
    ("Por nacionalidad", "Nacionalidad extranjera", change("Extranjera", "Total")),
    ("Cruce", "Españoles nacidos en España", change("Española", "España")),
    ("Cruce", "Españoles nacidos en el extranjero", change("Española", "Extranjero")),
]
labels = [r[1] for r in rows][::-1]
vals = [r[2] for r in rows][::-1]
fig = new_figure(height=380)
fig.add_bar(y=labels, x=vals, orientation="h", marker_color=[ACCENT if v > 0 else PALETTE[1] for v in vals],
            text=[f"{v:+.2f}".replace(".", ",") for v in vals], textposition="outside")
fig.update_layout(xaxis_title="Millones de personas", xaxis=dict(tickformat=",.1f", zeroline=True, range=[-1.8, 4.8]), showlegend=False)
add_source(fig, SOURCE)
place_source(fig, y=-0.3, bottom=110)
fig.show()

::: {.resultado}
**A2 se sostiene con matices.** Con el país de nacimiento (A2a) se cumple: −0,95 millones de nacidos
en España y +4,12 millones de nacidos fuera. Con la nacionalidad (A2b) no: la población con
nacionalidad española **aumentó** en 0,34 millones, porque los españoles nacidos en el extranjero
—nacionalizados, pero también retornados e hijos de españoles nacidos fuera— crecieron en 1,34
millones, más de lo que cayeron los
españoles nacidos en España (−1,00 millones).
:::

### Sin población de origen extranjero, el saldo natural sería aún más negativo

«Sin inmigración, España perdería población» es una afirmación **contrafactual**: no se puede
observar. Lo que sí muestran los datos es que la población extranjera, joven en promedio, aporta
una parte creciente de los nacimientos y pocas defunciones.

In [10]:
#| label: fig-madres
#| fig-cap: "Izquierda: porcentaje de nacimientos de madre con nacionalidad extranjera y de madre nacida en el extranjero, 2002–2024 (nacida en el extranjero: desde 2009; nacionalidad 2002–2008 estimada con el porcentaje de la tabla 2776 del INE, que da valores 0,1–0,3 puntos más bajos). Derecha: saldo natural por nacionalidad, 2009–2024 (nacimientos según la nacionalidad de la madre menos defunciones según la del fallecido)."
b = births.with_columns(
    pct_nac=100 * pl.col("madre_nacionalidad_extranjera") / pl.col("nacimientos"),
    pct_nacida=100 * pl.col("madre_nacida_extranjero") / pl.col("nacimientos"),
)
nb = (births.join(deaths, on="anio").with_columns(
    nat_esp=pl.col("nacimientos") - pl.col("madre_nacionalidad_extranjera") - pl.col("def_nacionalidad_espanola"),
    nat_ext=pl.col("madre_nacionalidad_extranjera") - pl.col("def_nacionalidad_extranjera"),
))
fig = make_subplots(rows=1, cols=2, subplot_titles=("Nacimientos de madre extranjera (%)", "Saldo natural por nacionalidad (personas)"))
fig.add_scatter(x=b["anio"], y=b["pct_nac"], name="Madre de nacionalidad extranjera", mode="lines+markers", line=dict(color=ACCENT), row=1, col=1)
fig.add_scatter(x=b["anio"], y=b["pct_nacida"], name="Madre nacida en el extranjero", mode="lines+markers", line=dict(color=PALETTE[2]), row=1, col=1)
fig.add_bar(x=nb["anio"], y=nb["nat_esp"], name="Nacionalidad española", marker_color=PALETTE[1], row=1, col=2)
fig.add_bar(x=nb["anio"], y=nb["nat_ext"], name="Nacionalidad extranjera", marker_color=ACCENT, opacity=0.6, row=1, col=2)
fig.update_yaxes(tickformat=",.0f")
fig.update_layout(height=430, barmode="relative", margin=dict(t=50),
                  legend=dict(orientation="h", yanchor="top", y=-0.12, x=0))
add_source(fig, SOURCE)
place_source(fig, y=-0.45, bottom=150)
fig.show()

In [11]:
#| include: false
r24 = nb.filter(pl.col("anio") == 2024).row(0, named=True)
pct_nacida_24 = 100 * r24["madre_nacida_extranjero"] / r24["nacimientos"]
pct_nac_24 = 100 * r24["madre_nacionalidad_extranjera"] / r24["nacimientos"]

::: {.resultado}
En 2024, uno de cada cuatro nacimientos fue de madre extranjera (25,6 %) y uno de cada tres de madre
nacida en el extranjero (33,3 %). Separando por nacionalidad, el saldo natural de la población
española fue de unas −179.000 personas y el de la población extranjera de unas +61.000: sin la
segunda, el saldo natural de España habría sido aún más negativo que el observado (−118.113).
:::

::: {.callout-note collapse="true" title="Por qué esta cuenta es solo aproximada"}
El nacimiento se asigna a la nacionalidad de la **madre**, no a la del niño, que puede ser española
si lo es el padre. Y la cuenta no dice qué habría pasado sin inmigración: sin ella, la población, su
edad y su fecundidad habrían sido otras. Muestra la contribución observada de cada grupo, no un
escenario alternativo.
:::

### ¿Cambia la tendencia con cada gobierno?

La figura de los componentes marca los periodos de gobierno. La tabla resume el promedio anual de
cada componente en cada etapa.

In [12]:
#| label: tbl-gobiernos
#| tbl-cap: "Promedio anual de los componentes del crecimiento por etapa de gobierno (personas). Cada año se asigna al gobierno que lo ocupó en su mayor parte (2018, el año de la moción de censura, cuenta para Sánchez). Antes de 2008 y en 2025, saldo migratorio implícito, que incluye el residuo; por eso en esas etapas saldo natural + saldo migratorio no siempre igualan el crecimiento. La etapa de Aznar solo cubre dos años de la serie."
#| code-fold: true
g = comp.filter(pl.col("anio").is_between(2002, 2025)).with_columns(
    gobierno=pl.col("anio").map_elements(lambda y: next(n for n, a, b in GOV if a <= y <= b), return_dtype=pl.String),
    mig=pl.coalesce("saldo_migratorio", "saldo_migratorio_implicito"),
)
show(g.group_by("gobierno", maintain_order=True)
     .agg(Años=pl.format("{}–{}", pl.col("anio").min(), pl.col("anio").max()),
          **{"Saldo natural": pl.col("saldo_natural").mean(),
             "Saldo migratorio": pl.col("mig").mean(),
             "Crecimiento": pl.col("crecimiento").mean()})
     .rename({"gobierno": "Gobierno"}))

| Gobierno | Años | Saldo natural | Saldo migratorio | Crecimiento |
|:---|:---|---:|---:|---:|
| Aznar | 2002–2003 | 53.640 | 702.451 | 756.092 |
| Zapatero | 2004–2011 | 101.557 | 373.035 | 533.845 |
| Rajoy | 2012–2017 | 14.185 | −40.021 | −28.858 |
| Sánchez | 2018–2025 | −108.882 | 473.258 | 368.929 |

::: {.aviso}
**Coincidir no es causar.** Las etapas coinciden con el ciclo económico: la expansión hasta 2008,
la crisis y la emigración de 2010–2015, la recuperación, la pandemia de 2020 y la expansión
posterior. También influyen los flujos internacionales (por ejemplo, desde Venezuela y Colombia) y
las políticas tienen efectos con retraso. Esta tabla describe **cuándo** cambió la tendencia, no
**por qué**; atribuirlo a un gobierno exigiría un diseño causal que esta entrada no tiene.
:::

### ¿Sigue la inmigración al ciclo económico?

La interpretación habitual es que la inmigración responde al **empleo**: llegan más personas cuando
se crea trabajo y menos (o se van) cuando se destruye. Es una hipótesis falsable: si fuera cierta,
los años de más saldo migratorio deberían ser también años de más creación de empleo.

::: {.callout-note collapse="true" title="Conceptos: la EPA, ocupados y tasa de paro"}
- La **Encuesta de Población Activa (EPA)** del INE se hace cada trimestre sobre una muestra de unas
  65.000 viviendas (unas 47.500 entrevistadas). Clasifica a las personas de 16 años o más en
  **ocupadas** (trabajaron al menos una hora en la semana de referencia, o tenían trabajo y estaban
  ausentes temporalmente, como los trabajadores en ERTE en 2020), **paradas** (sin trabajo,
  buscándolo y disponibles) e **inactivas**.
- **Activos** = ocupados + parados. **Tasa de paro** = parados / activos × 100.
- Es una **encuesta**: sus cifras tienen error de muestreo, pequeño en los totales nacionales.
- Tiene **cambios de método**: un nuevo cuestionario en 2005 y una nueva base de población (Censo
  2021) aplicada desde 2021 sin revisar los años anteriores. Pueden mover algunas variaciones
  anuales; por ejemplo, el aumento de 2005 es el mayor de la serie.
- Aquí se usan **medias anuales** de los cuatro trimestres, para compararlas con los saldos
  anuales.
:::

**Cómo lo medimos.** Para cada año $t$ comparamos el saldo migratorio con la **variación del
empleo de nacionalidad española**: la media de ocupados españoles del año $t$ menos la del año
$t-1$. Se usa el empleo de los **españoles** y no el total porque el total incluye a los propios
inmigrantes que encuentran trabajo: la relación sería en parte mecánica. La EPA separa a quienes
tienen **doble nacionalidad**; se excluyen, porque muchos son personas nacionalizadas que
conservan su nacionalidad de origen, y así se reduce (sin eliminarlo) el efecto de las
nacionalizaciones. La asociación se mide con el **coeficiente de correlación de Pearson**, $r$.

::: {.callout-note collapse="true" title="El coeficiente de correlación de Pearson"}
Para $n$ pares $(x_t, y_t)$,
$$
r = \frac{\sum_t (x_t - \bar x)(y_t - \bar y)}{\sqrt{\sum_t (x_t - \bar x)^2}\,\sqrt{\sum_t (y_t - \bar y)^2}} \in [-1, 1].
$$
Mide la **asociación lineal**: $r$ cerca de 1, los años con $x$ alto tienden a tener $y$ alto;
cerca de 0, no hay relación lineal; cerca de −1, la relación es inversa. No dice nada de causas, y
con series temporales hay que tener cuidado: dos series con tendencia se correlacionan aunque no
tengan relación. Por eso se correlacionan **variaciones** del empleo, no niveles.
:::

**Criterios, fijados (y anotados en el plan) antes de descargar la EPA.** Con $r$ entre el saldo migratorio del año
$t$ y la variación del empleo español del mismo año (2003–2025): *se sostiene* si $r \ge 0{,}7$;
*con matices* si $0{,}4 \le r < 0{,}7$; *no se sostiene* si $r < 0{,}4$. Los umbrales son una
convención, no un resultado estadístico. Como comprobaciones se calculan también $r$ con el empleo
del año anterior ($t-1$), $r$ incluyendo la doble nacionalidad, $r$ solo con los años de saldo
migratorio publicado (2008–2024) y $r$ con la tasa de paro (en nivel y en variación).

In [13]:
#| label: epa-prepare
#| include: false
from common.paths import data_dirs
HAS_EPA = (data_dirs(TOPIC)[1] / "p01_epa.parquet").exists()
SOURCE_EPA = "Fuente: INE (EPA, ECP, MNP, Estadística de Migraciones). Elaboración propia."
fmt_r = lambda r: f"{r:.2f}".replace(".", ",").replace("-", "−")
if HAS_EPA:
    epa = read_processed(TOPIC, "p01_epa")
    emp = (epa.pivot(on="nacionalidad", index="anio", values=["ocupados", "tasa_paro"])
              .sort("anio")
              .with_columns(d_ocup_esp=pl.col("ocupados_española").diff(),
                            d_ocup_esp_prev=pl.col("ocupados_española").diff().shift(1),
                            d_ocup_espdob=(pl.col("ocupados_española") + pl.col("ocupados_doble")).diff(),
                            d_paro=pl.col("tasa_paro_total").diff()))
    cyc = (comp.select("anio", "saldo_migratorio",
                       mig=pl.coalesce("saldo_migratorio", "saldo_migratorio_implicito"))
               .join(emp, on="anio", how="inner")
               .filter(pl.col("anio").is_between(2003, 2025), pl.col("d_ocup_esp").is_not_null()))
    corr = lambda d, a, b: d.drop_nulls([a, b]).select(pl.corr(a, b)).item()
    r_main = corr(cyc, "mig", "d_ocup_esp")
    r_lag = corr(cyc, "mig", "d_ocup_esp_prev")
    r_off = corr(cyc.filter(pl.col("saldo_migratorio").is_not_null()), "saldo_migratorio", "d_ocup_esp")
    r_paro = corr(cyc, "mig", "tasa_paro_total")
    r_dparo = corr(cyc, "mig", "d_paro")
    r_dob = corr(cyc, "mig", "d_ocup_espdob")
    r_noboom = corr(cyc.filter(pl.col("anio") > 2007), "mig", "d_ocup_esp")
    y0, y1, n_cyc = cyc["anio"].min(), cyc["anio"].max(), cyc.height

In [14]:
#| label: fig-ciclo
#| fig-cap: "Arriba: saldo migratorio anual (antes de 2008 y en 2025, implícito). Abajo: variación anual del número medio de ocupados de nacionalidad española (EPA). Miles de personas."
if HAS_EPA:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=("Saldo migratorio", "Variación de los ocupados españoles"))
    fig.add_bar(x=cyc["anio"], y=cyc["mig"] / 1e3, marker_color=ACCENT, name="Saldo migratorio", row=1, col=1)
    fig.add_bar(x=cyc["anio"], y=cyc["d_ocup_esp"], marker_color=PALETTE[1], name="Δ ocupados españoles", row=2, col=1)
    fig.update_yaxes(title_text="Miles", tickformat=",.0f")
    fig.update_layout(height=480, showlegend=False, margin=dict(t=40))
    add_source(fig, SOURCE_EPA)
    place_source(fig, y=-0.08, bottom=70)
    fig.show()
else:
    display(Markdown("*Pendiente: ejecutar la descarga y el procesado de la EPA.*"))

In [15]:
#| label: fig-ciclo-dispersion
#| fig-cap: "Cada punto es un año: saldo migratorio frente a la variación de los ocupados de nacionalidad española (miles de personas)."
if HAS_EPA:
    fig = new_figure(y_title="Variación de los ocupados españoles (miles)", height=440)
    fig.add_scatter(x=cyc["mig"] / 1e3, y=cyc["d_ocup_esp"], mode="markers+text",
                    text=cyc["anio"].cast(pl.String), textposition="top center",
                    textfont=dict(size=10, color=GREY), marker=dict(color=ACCENT, size=8), showlegend=False)
    fig.update_layout(xaxis_title="Saldo migratorio (miles)", xaxis=dict(tickformat=",.0f"),
                      yaxis=dict(tickformat=",.0f"))
    add_source(fig, SOURCE_EPA)
    place_source(fig, y=-0.2, bottom=90)
    fig.show()

In [16]:
#| echo: false
if HAS_EPA:
    verdict = ("se sostiene" if r_main >= 0.7 else "se sostiene con matices" if r_main >= 0.4 else "no se sostiene")
    display(Markdown(f"""
::: {{.resultado}}
**La hipótesis {verdict}** con el criterio fijado. Entre {y0} y {y1} ({n_cyc} años), la correlación
entre el saldo migratorio y la variación del empleo español del mismo año es $r = {fmt_r(r_main)}$.
Comprobaciones: con el empleo del año anterior, $r = {fmt_r(r_lag)}$; incluyendo la doble
nacionalidad, $r = {fmt_r(r_dob)}$; solo con saldo migratorio publicado (2008–2024),
$r = {fmt_r(r_off)}$; con la variación de la tasa de paro, $r = {fmt_r(r_dparo)}$ (con su nivel,
$r = {fmt_r(r_paro)}$). Sin los años de expansión 2003–2007, $r = {fmt_r(r_noboom)}$: la asociación
descansa sobre todo en esa etapa.
:::
"""))


::: {.resultado}
**La hipótesis se sostiene con matices** con el criterio fijado. Entre 2003 y 2025 (23 años), la correlación
entre el saldo migratorio y la variación del empleo español del mismo año es $r = 0,55$.
Comprobaciones: con el empleo del año anterior, $r = 0,71$; incluyendo la doble
nacionalidad, $r = 0,56$; solo con saldo migratorio publicado (2008–2024),
$r = 0,41$; con la variación de la tasa de paro, $r = −0,38$ (con su nivel,
$r = −0,94$). Sin los años de expansión 2003–2007, $r = 0,43$: la asociación
descansa sobre todo en esa etapa.
:::


In [17]:
#| include: false
# Comprobaciones de las afirmaciones de la lectura (si fallan, hay que reescribirla)
if HAS_EPA:
    v = {r["anio"]: r for r in cyc.iter_rows(named=True)}
    assert all(v[y]["mig"] > 500_000 and v[y]["d_ocup_esp"] > 0 for y in range(2003, 2008))
    assert all(v[y]["d_ocup_esp"] < 0 for y in range(2008, 2014)) and all(v[y]["mig"] < 0 for y in range(2010, 2016))
    assert v[2014]["d_ocup_esp"] > 0 and v[2015]["d_ocup_esp"] > 0
    assert v[2020]["d_ocup_esp"] < 0 < v[2020]["mig"]
    assert all(v[y]["d_ocup_esp"] > 0 and v[y]["mig"] > 0 for y in list(range(2016, 2020)) + list(range(2022, 2026)))
    assert v[2008]["d_ocup_esp"] < 0 < v[2008]["mig"] and v[2009]["mig"] > 0
    assert r_lag > r_main and r_paro < -0.8 and r_noboom < r_main
    assert 0.4 <= r_main < 0.7  # el veredicto «con matices» de la tabla final

**Lectura.**

- **En las grandes etapas, la relación es clara.** En 2003–2007 se creaba empleo y el saldo
  migratorio superaba el medio millón al año; entre 2008 y 2013 se destruyó empleo español todos
  los años y entre 2010 y 2015 el saldo migratorio fue negativo; en 2016–2019 y desde 2022
  coinciden creación de empleo y saldo migratorio positivo.
- **Año a año, no tanto.** En los cambios de etapa los datos **sugieren un retraso**: en 2008 y
  2009 ya se destruía empleo y el saldo migratorio seguía siendo positivo; en 2014 y 2015 el empleo
  español ya crecía y el saldo seguía siendo negativo. Con el empleo del año anterior la
  correlación sube, pero esa comprobación no era el criterio fijado, está justo en el umbral y no
  cambia el veredicto: es una pista, no un resultado.
- **La asociación depende sobre todo de la expansión de 2003–2007**: sin esos años, $r$ baja
  claramente. Y en 2020 la pandemia destruyó empleo mientras el saldo migratorio seguía positivo.
- **La tasa de paro**: con su nivel, $r$ es cercano a −1, pero esa cifra está inflada, porque
  compara dos magnitudes que cambian despacio (el caso que advierte el desplegable sobre $r$). Con
  su variación anual, la asociación es mucho más débil.

::: {.aviso}
**Lo que esto no dice.** Una correlación entre dos series anuales no indica qué causa qué: el
empleo puede atraer inmigración, la inmigración puede crear empleo (más consumo, más empresas) y
ambas pueden responder a un tercer factor, como el ciclo de la construcción antes de 2008. Además,
las nacionalizaciones trasladan cada año ocupados extranjeros al grupo «español», lo que engorda la
variación del empleo español en los años con muchas nacionalizaciones (2023–2025). Y con 23
años, cada año pesa mucho en $r$.
:::

## Veredicto

| Afirmación | Veredicto | Por qué |
|---|---|---|
| **A1.** La población crece solo por la llegada de población del extranjero | **Se sostiene** (desde 2016) | Saldo natural negativo desde 2015; en 2016–2025, crecimiento siempre inferior al saldo migratorio; población nacida en España en descenso cada año. En 2015 la población aún bajó. |
| **A2.** La población de origen español disminuye y la de origen extranjero aumenta | **Se sostiene con matices** | Cierta por país de nacimiento (−0,95 M nacidos en España). Falsa por nacionalidad: la población con nacionalidad española creció +0,34 M, por las nacionalizaciones. |
| **H.** La inmigración va acompasada con el ciclo del empleo (hipótesis de la interpretación habitual) | **Se sostiene con matices** | Coinciden las grandes etapas (expansión, crisis, recuperación), pero año a año la correlación es moderada ($r \approx 0{,}55$, en la parte baja del tramo) y descansa sobre todo en 2003–2007; en los cambios de etapa los datos sugieren un retraso. Asociación, no causa. |

## Interpretación

- **Hipótesis: el saldo natural negativo es estructural.** La explicación demográfica habitual es la
  baja fecundidad sostenida y el envejecimiento: generaciones menos numerosas en edad de tener hijos
  y más numerosas en edad avanzada. Esta entrada no lo contrasta (pendiente: referencias y datos
  de fecundidad y estructura por edad).
- **La inmigración y el ciclo del empleo.** La sección anterior contrasta esta hipótesis con la
  EPA: es una asociación entre series anuales, no una relación causal.
- **La nacionalidad y el país de nacimiento cuentan historias distintas**, y ambas son correctas: la
  primera refleja la integración jurídica (las nacionalizaciones), la segunda la trayectoria vital.
  Cuál usar depende de la pregunta.

## Limitaciones

- Datos de 2025 de nacimientos y defunciones: estimaciones experimentales; saldo migratorio de 2025
  y anterior a 2008: implícito.
- Cambio de estadística de migraciones en 2021 y residuo de la ecuación compensadora en 2008–2011.
- El análisis es descriptivo: no atribuye las tendencias a ningún gobierno ni a ninguna política.
- Solo escala nacional; las diferencias entre comunidades quedan para otra entrada.
- La relación con el empleo es una correlación entre dos series de 23 datos anuales: no identifica
  causas y es sensible a años concretos y a la serie de saldo migratorio usada (implícito o
  publicado). La EPA cambió de cuestionario en 2005 y de base de población en 2021.

## Fuentes, técnicas y código

- **Fuentes:** INE, *Estadística Continua de Población*, *Movimiento Natural de la Población*,
  *Estadística de Migraciones* y *Estadística de Migraciones y Cambios de Residencia*,
  *Indicadores Demográficos Básicos*, *Encuesta de Población Activa* y estimaciones experimentales
  EMN y EDeS. Descarga: 15-XI-2026 (EPA: 16-XI-2026).
- **Técnicas:** frecuencias absolutas y relativas, variaciones y medias anuales, la ecuación
  compensadora como identidad contable · {{< var materiales.desc-2-1 >}}; coeficiente de
  correlación de Pearson · *Estadística descriptiva*, parte 3.2 (en preparación)
- **Código:** descarga y procesado en `poblacion/src/` del
  [repositorio](https://github.com/FabioScielzoOrtiz/la-mirada-del-dato).

## Referencias

<!-- Correcciones: AAAA-MM-DD · qué se corrigió -->

::: {.content-hidden}
## Zona de trabajo (no se publica)

Pendiente antes de publicar:

- Verificar todas las citas en su fuente primaria (vídeo, programa, transcripción).
- Buscar declaraciones recientes de Sumar, Podemos, sindicatos y CEOE sobre demografía.
- Ciclo económico: citar las notas del INE sobre los cambios de la EPA (cuestionario 2005,
  base Censo 2021); decidir si se añade la comparación con la UE (Eurostat); referencias sobre
  migración y ciclo en España.
- Referencias académicas para la interpretación (fecundidad, envejecimiento, migración y ciclo).
- Verificación independiente de cifras y citas.

In [18]:
# Comprobaciones rápidas (no se publican)
print(f"Saldo natural ≤ 0 en {dec_ok_nat} de 10 años; crecimiento ≤ saldo migratorio en {dec_ok_mig} de {dec_n_mig}; 2016: {n2016:.0f}")
print(f"Nacidos en España: {e16:,.0f} → {e26:,.0f} ({e26 - e16:+,.0f}); baja todos los años: {esp_down_every_year}")
print(f"Nacidos fuera: {x16:,.0f} → {x26:,.0f} ({x26 - x16:+,.0f})")
print(f"2024: % madre extranjera (nacionalidad) {pct_nac_24:.1f}; nacida fuera {pct_nacida_24:.1f}; saldo natural españoles {r24['nat_esp']:,.0f}; extranjeros {r24['nat_ext']:,.0f}")

Saldo natural ≤ 0 en 10 de 10 años; crecimiento ≤ saldo migratorio en 9 de 9; 2016: -28
Nacidos en España: 40,505,719 → 39,560,145 (-945,574); baja todos los años: True
Nacidos fuera: 5,913,165 → 10,036,354 (+4,123,189)
2024: % madre extranjera (nacionalidad) 25.6; nacida fuera 33.3; saldo natural españoles -178,993; extranjeros 60,887


:::